# Notebook 1: Base Stack Export

Exports a single multi-band raster at MODIS sinusoidal resolution (~463m) per basin.
This asset contains ALL variables needed for downstream FRIP and GEDI analysis.

**Run once.** Downstream notebooks load this asset.

**Key design decisions (from GEE best practices + trial and error):**
- No `reproject()` — export's `scale`/`crs` defines the output grid
- Each band has its own independent `reduceResolution` chain — no cross-dataset dependencies
- Per-basin exports — avoid spanning the Atlantic in one bounding box
- Masking deferred — `forest_fraction` exported as continuous band, threshold applied downstream

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
GEE_PROJECT = 'quantum-bonus-434714-t2'
ASSET_ROOT = f'projects/{GEE_PROJECT}/assets/DefaunationFromSpace'

# Study Regions — per-basin exports avoid spanning the Atlantic
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Years
YEARS = list(range(2001, 2024))  # 2001-2023 inclusive

# GEE Dataset IDs
MODIS_NPP     = 'MODIS/061/MOD17A3HGF'
GLOFAS        = 'JRC/CEMS_GLOFAS/FloodHazard/v1'
MERIT_HYDRO   = 'MERIT/Hydro/v1_0_1'
FOREST_MASK   = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS  = 10   # Undisturbed since ~1982
SRTM          = 'USGS/SRTMGL1_003'
GEDI_L2B      = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
CHIRPS        = 'UCSB-CHG/CHIRPS/DAILY'
SOILGRIDS_CLAY = 'OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02'

# GEDI date range
GEDI_START = '2020-01-01'
GEDI_END   = '2023-12-31'

# CHIRPS date range (climatological mean)
PRECIP_START = '2001-01-01'
PRECIP_END   = '2023-12-31'

print("\u2713 Configuration loaded.")
print(f"  Basins: {[b[0] for b in BASINS]}")
print(f"  Years: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} years)")
print(f"  Asset root: {ASSET_ROOT}")

In [ ]:
# =============================================================================
# BLOCK 2: BUILD BAND FUNCTIONS
# =============================================================================
# Each function returns an ee.Image band (or set of bands) with
# setDefaultProjection applied so reduceResolution knows the input grid.
# NO reproject() anywhere. The export's scale/crs defines the output.

# --- MODIS NPP Reference Projection ---
# All bands will be exported at this resolution and CRS.
_modis_col = ee.ImageCollection(MODIS_NPP).select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = MODIS_PROJ.nominalScale()

def build_npp_bands():
    """MODIS NPP: median + 23 annual bands. Already at native MODIS resolution."""
    modis = ee.ImageCollection(MODIS_NPP).select('Npp')
    
    # Annual images via filter (not toList — avoids breaking GEE optimizations)
    def get_annual(year):
        year = ee.Number(year)
        return modis.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).first().set('year', year)
    
    annual_imgs = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(get_annual)
    )
    
    # Median NPP
    median_npp = annual_imgs.median().rename('Npp_median')
    
    # Annual NPP as multi-band image
    annual_npp = annual_imgs.toBands()
    band_names = [f'NPP_{y}' for y in YEARS]
    annual_npp = annual_npp.rename(band_names)
    
    return median_npp, annual_npp

def build_flood_frequency():
    """Flood frequency: count of return periods where flooding occurs.
    
    Legacy pattern: binary .gte(0) per return period, summed.
    Masked to hydrologically connected areas (MERIT HND > 0).
    """
    glofas = ee.ImageCollection(GLOFAS)
    periods = [10, 20, 50, 75, 100, 200, 500]
    
    def get_flood_binary(period):
        period = ee.Number(period)
        return glofas.filter(
            ee.Filter.eq('return_period', period)
        ).mosaic().gte(0).set('return_period', period)
    
    flood_images = ee.List(periods).map(get_flood_binary)
    flood_proj = ee.Image(glofas.first()).projection()
    
    # Sum across return periods, unmask to 0 where no data
    flood_freq = ee.ImageCollection.fromImages(flood_images).sum().unmask()
    
    # Set native projection and mask to hydrologically connected
    hnd_mask = ee.Image(MERIT_HYDRO).select('hnd').gt(0)
    flood_freq = flood_freq.setDefaultProjection(crs=flood_proj).updateMask(hnd_mask)
    
    # Aggregate to MODIS grid
    flood_reduced = flood_freq.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('flood_freq')
    
    return flood_reduced

def build_forest_fraction():
    """JRC TMF forest fraction at MODIS resolution.
    
    Exported as continuous 0-1 value. Thresholding (>=0.95) applied downstream.
    """
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf_proj = tmf_col.first().projection()
    
    forest = tmf_col.mosaic().eq(FOREST_CLASS).setDefaultProjection(tmf_proj)
    
    forest_frac = forest.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('forest_fraction')
    
    return forest_frac

def build_terrain():
    """Elevation and slope from SRTM, aggregated to MODIS resolution."""
    srtm = ee.Image(SRTM)
    srtm_proj = srtm.select('elevation').projection()
    
    elev = srtm.select('elevation').setDefaultProjection(srtm_proj)
    elev_reduced = elev.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('elevation')
    
    slp = ee.Terrain.slope(srtm).setDefaultProjection(srtm_proj)
    slp_reduced = slp.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('slope')
    
    return elev_reduced, slp_reduced

def build_hnd():
    """Height Above Nearest Drainage from MERIT Hydro."""
    hnd = ee.Image(MERIT_HYDRO).select('hnd')
    hnd_proj = hnd.projection()
    
    hnd_reduced = hnd.setDefaultProjection(hnd_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('hnd')
    
    return hnd_reduced

def build_gedi():
    """GEDI L2B: UOI mean, observation count N, and rh98 canopy height.
    
    Each band reduced independently (UOI+rh98 = mean, N = sum).
    No external masks — masking deferred to downstream.
    """
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(GEDI_START, GEDI_END)
    native_proj = gedi.first().projection()
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        return uoi.clamp(0, 1).rename('UOI')
    
    uoi_col = gedi.map(calc_uoi)
    
    # UOI mean
    uoi_mean = uoi_col.mean().rename('GEDI_UOI').setDefaultProjection(native_proj)
    uoi_reduced = uoi_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    )
    
    # Observation count
    uoi_count = uoi_col.count().rename('GEDI_N').setDefaultProjection(native_proj)
    n_reduced = uoi_count.reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    )
    
    # Canopy height (rh98)
    rh98_mean = gedi.select('rh98').mean().rename('GEDI_rh98').setDefaultProjection(native_proj)
    rh98_reduced = rh98_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    )
    
    return uoi_reduced, n_reduced, rh98_reduced

def build_precip():
    """Mean annual precipitation from CHIRPS."""
    chirps = ee.ImageCollection(CHIRPS).filterDate(PRECIP_START, PRECIP_END)
    chirps_proj = chirps.first().projection()
    
    # Sum daily precip per year, then take the mean across years
    def annual_total(year):
        year = ee.Number(year)
        return chirps.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).sum().set('year', year)
    
    annual_precip = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(annual_total)
    )
    mean_precip = annual_precip.mean().rename('precip').setDefaultProjection(chirps_proj)
    
    # CHIRPS is ~5km (coarser than MODIS) so no reduceResolution needed.
    # The export will resample to MODIS grid via nearest neighbor.
    return mean_precip

def build_clay():
    """Soil clay fraction from OpenLandMap/SoilGrids (0-200mm depth)."""
    clay = ee.Image(SOILGRIDS_CLAY)
    # Use mean of available depth layers
    clay_mean = clay.reduce(ee.Reducer.mean()).rename('clay')
    clay_proj = clay.select(0).projection()
    
    clay_reduced = clay_mean.setDefaultProjection(clay_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    )
    
    return clay_reduced

print("\u2713 All band-building functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: ASSEMBLE FULL STACK
# =============================================================================

def build_full_stack():
    """Assembles all 34 bands into a single ee.Image for export.
    
    Each band is computed independently — no cross-band dependencies.
    The export's scale/crs handles the output grid alignment.
    
    Returns: (stack, modis_proj, modis_scale)
    """
    print("  Building NPP bands...")
    npp_median, npp_annual = build_npp_bands()
    
    print("  Building flood frequency...")
    flood = build_flood_frequency()
    
    print("  Building forest fraction...")
    forest = build_forest_fraction()
    
    print("  Building terrain (elevation + slope)...")
    elev, slope = build_terrain()
    
    print("  Building HND...")
    hnd = build_hnd()
    
    print("  Building GEDI (UOI + N + rh98)...")
    uoi, n, rh98 = build_gedi()
    
    print("  Building precipitation...")
    precip = build_precip()
    
    print("  Building clay...")
    clay = build_clay()
    
    # Stack all bands — order matters for downstream band selection
    stack = ee.Image.cat([
        npp_median,     # 1:  Npp_median
        npp_annual,     # 2-24: NPP_2001 ... NPP_2023
        flood,          # 25: flood_freq
        forest,         # 26: forest_fraction
        elev,           # 27: elevation
        slope,          # 28: slope
        hnd,            # 29: hnd
        uoi,            # 30: GEDI_UOI
        n,              # 31: GEDI_N
        rh98,           # 32: GEDI_rh98
        precip,         # 33: precip
        clay,           # 34: clay
    ]).toFloat()  # Export as float32 (not float64) to halve asset size
    
    print("\u2713 Full stack assembled.")
    return stack

stack = build_full_stack()
band_names = stack.bandNames().getInfo()
print(f"\n  Total bands: {len(band_names)}")
print(f"  Bands: {band_names}")

In [ ]:
# =============================================================================
# BLOCK 4: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running unit tests on assembled stack...\n")
    passed = 0
    
    # Test 1: Band count
    bands = stack.bandNames().getInfo()
    expected = 34
    assert len(bands) == expected, f"Expected {expected} bands, got {len(bands)}: {bands}"
    passed += 1
    print(f"  [1/4] \u2713 Band count: {len(bands)} bands")
    
    # Test 2: Key bands present
    required = ['Npp_median', 'NPP_2001', 'NPP_2023', 'flood_freq',
                'forest_fraction', 'elevation', 'slope', 'hnd',
                'GEDI_UOI', 'GEDI_N', 'GEDI_rh98', 'precip', 'clay']
    missing = [b for b in required if b not in bands]
    assert not missing, f"Missing bands: {missing}"
    passed += 1
    print(f"  [2/4] \u2713 All required bands present")
    
    # Test 3: Sample a small region (quick server-side check)
    test_point = ee.Geometry.Point([20, 0])  # Congo interior
    sample = stack.sample(region=test_point, scale=500, numPixels=1).first()
    props = sample.toDictionary().getInfo()
    non_null = {k: v for k, v in props.items() if v is not None}
    passed += 1
    print(f"  [3/4] \u2713 Server-side sample: {len(non_null)}/{len(props)} non-null bands at Congo test point")
    
    # Test 4: MODIS projection info
    proj_info = MODIS_PROJ.getInfo()
    scale_val = MODIS_SCALE.getInfo()
    passed += 1
    print(f"  [4/4] \u2713 Export projection: {proj_info['crs']} at {scale_val:.1f}m")
    
    print(f"\n{'='*60}")
    print(f"  \u2713 ALL {passed} TESTS PASSED")
    print(f"  Ready to export. Call export_base_stacks(dry_run=False)")
    print(f"{'='*60}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 5: EXPORT
# =============================================================================

def safe_start(task, asset_id):
    """Delete existing asset if present, then start the export task."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_base_stacks(dry_run=True):
    """Export the base stack as one asset per basin.
    
    Exports 2 tasks total:
      BaseStack_Congo  (34 bands at ~463m MODIS sinusoidal)
      BaseStack_Amazon (34 bands at ~463m MODIS sinusoidal)
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        asset_id = f'{ASSET_ROOT}/BaseStack_{basin_name}'
        
        task = ee.batch.Export.image.toAsset(
            image=stack,
            description=f'BaseStack_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs=MODIS_PROJ.crs(),
            maxPixels=1e13
        )
        tasks.append((task, asset_id))
    
    print(f"\u2713 {len(tasks)} export tasks configured:")
    for _, aid in tasks:
        print(f"    {aid}")
    
    if dry_run:
        print("\nDRY RUN. Call export_base_stacks(dry_run=False) to start.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f"  \u2713 Started: {asset_id.split('/')[-1]}")
        print("\n\u2713 All tasks started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")
        print("  Wait for completion before running NB2.")

export_base_stacks(dry_run=True)